# Tutorial: Survey Weights in SocialMix

Author: Shozen Dan (2026/11/12)

## Introduction
Social contact surveys ask respondents to record the contacts they had on a given day. A contact matrix is an *average* of the contact behaviour captured in the data. For instance, an age-stratified contact matrix encodes the average number of contacts by age of respondent and age of contact.

Various factors influence contact behaviour. For instance, the contact behaviour during weekdays where children go to school and adults go to work will be different from weekends and holidays. If the ratio between respondents who reported their contact on weekdays and those on weekends is not 5:2, then a naive average could potentially bias the results in favour of one or the other.

In general, weighting comes into play when we believe that certain factors such as sex, day-of-week, and socioeconomic status influence contact behaviour and we wish to average over those factors. We weight when the composition of our sample is not, in some sense, *representative* of the true composition when we stratify by these factors.

This tutorial aims to be a one-stop guide to applying weights to contact matrices with the `SocialMix` class in `cntmosaic`.

**Prerequisites.** [Tutorial 02](02_Tutorial_SocialMix.ipynb) (fitting `SocialMix`).

## Contents

1. [When to apply weights?](#1.-When-to-apply-weights?)
2. [Data and the unweighted baseline](#2.-Data-and-the-unweighted-baseline)
3. [Day-of-week weights](#3.-Day-of-week-weights)
4. [Matching 5:2 within age groups or across the sample?](#4.-Matching-5:2-within-age-groups-or-across-the-sample?)
5. [How does weighting work?](#5.-How-does-weighting-work?)
6. [Which weight factor should I use?](#6.-Which-weight-factor-should-I-use?)
7. [The weight factors](#7.-The-weight-factors)
8. [Inspecting and stabilising weights](#8.-Inspecting-and-stabilising-weights)
9. [Uncertainty with the weighted bootstrap](#9.-Uncertainty-with-the-weighted-bootstrap)
10. [Should we weight or stratify?](#10.-Should-we-weight-or-stratify?)
11. [Limitations and checklist](#11.-Limitations-and-checklist)
12. [Summary and further reading](#12.-Summary-and-further-reading)
- [Appendix: factor arguments and data assumptions](#Appendix:-factor-arguments-and-data-assumptions)

In [ ]:
# @title Setup (Google Colab only — skip if cntmosaic is already installed locally)
#
# Clones the cntmosaic repository, installs the package, and sets the working
# directory so that the relative data paths in this notebook resolve correctly.
import sys, os, subprocess

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab — cloning repository and installing cntmosaic...")
    subprocess.check_call(
        ["git", "clone", "--quiet", "https://github.com/MLGlobalHealth/cntmosaic.git"]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "./cntmosaic"]
    )
    os.chdir("cntmosaic/tutorials")
    print(f"Done. Working directory set to: {os.getcwd()}")

In [ ]:
import warnings

import altair as alt
import numpy as np
import pandas as pd

from cntmosaic.analysis import ModelSummariserSocialMix
from cntmosaic.dataloader import ContactData, ParticipantData, PopulationData
from cntmosaic.models import SocialMix
from cntmosaic.utils import AgeGroupSpecs
from cntmosaic.vis import plot_mosaic_pixilated
from cntmosaic.weights import (
    AgePostStratWeights,
    CategoryWeights,
    DayOfWeekWeights,
    DesignWeights,
    PostStratWeights,
    WeightSpecs,
    WeightWarning,
)

## 1. When to apply weights?

**Weighting is a choice.** Before we incorporate weights in our estimator, we must consider if weighting is appropriate or necessary. In short, apply weighting when both of the following points are true:
1. We wish to average over a factor (sex, day-of-week, socioeconomic status) that is believed to influence contact behaviour.
2. The sample composition is different from the true population composition when stratifying by that factor.

Weights supplied with the survey (design weights, e.g. for unequal selection probabilities or non-response) are a different case: they follow from how the sample was drawn rather than from a factor we choose to average over, and the survey documentation usually states how they should be used (Section 6).

It is important to keep in mind that **weighting changes what is being estimated.** Day-of-week weights, for example, target the average over a typical week. As we show in Section 10, this average **understates the contact intensity of children** during the school term. If the analysis concerns school-term transmission, it may in turn underestimate key epidemiological quantities such as $R_0$ and final size.

## 2. Data and the unweighted baseline

We use the POLYMOD United Kingdom survey (Mossong et al., 2008) as a running example.

In [ ]:
# Participants: single-year ages, sex, and the survey day
df_part = pd.read_csv("data/POLYMOD_UK_participant_common.csv")

# The survey day is stored in a separate file and merged into the participant data.
df_day = pd.read_csv("data/POLYMOD_UK_participant_dayofweek.csv")
df_part = df_part.merge(df_day, on="part_id", how="left", validate="one_to_one")

# Contacts: single-year contact ages (contacts without an age are dropped)
df_cnt = pd.read_csv("data/POLYMOD_UK_contact_common.csv")
df_cnt = df_cnt.rename(columns={"cnt_age_exact": "cnt_age"})

# Population: single years 0-89 (the open-ended "90+" row is left out)
df_pop = pd.read_csv("data/UK_population_2011.csv")
df_pop = df_pop[~df_pop["age"].str.endswith("+")].copy()
df_pop["age"] = df_pop["age"].astype(int)
df_pop["P"] = df_pop["Male"] + df_pop["Female"]
df_pop = df_pop[["age", "P"]]

# Display the first 5 rows
df_part.head()

Next, we look at how the survey days are distributed.

In [ ]:
day_type = pd.Series(
    np.select(
        [df_part["dayofweek"].isin([1, 2, 3, 4, 5]), df_part["dayofweek"].isin([0, 6])],
        ["weekday", "weekend"],
        default="missing",
    ),
    index=df_part["part_id"],
    name="day_type",
)
counts = day_type.value_counts()
share = counts["weekday"] / (counts["weekday"] + counts["weekend"])
print(
    f"Weekday share among participants with a known day: {share:.3f} (a week: 5/7 = {5 / 7:.3f})"
)
counts

Among the participants with a known survey day, 73.6% reported on a weekday, compared with $5/7 \approx 71.4\%$ of the days in a week. A naive average would therefore lean slightly towards weekday behaviour. 43 participants have no recorded day.

We now build the data containers as in Tutorial 02. The only new argument is `aux_cols`, which keeps the variables needed for weighting on the participant data. `aux_cols` takes the column names of the DataFrame and stores each column with the `part_` prefix (`dayofweek` becomes `part_dayofweek`). The weight factors accept a column name with or without the prefix, so `"gender"` and `"part_gender"` refer to the same column.

`ParticipantData` and `ContactData` report routine data notes as `UserWarning`s: here, that `part_gender` is stored as categorical and that 3 contacts without an age are dropped.

In [ ]:
age_group_specs = AgeGroupSpecs(0, 79, 5)  # 16 five-year age groups

part_data = ParticipantData(
    df_part,
    id_col="part_id",
    age_col="part_age",
    aux_cols=["dayofweek", "part_gender"],
)
cnt_data = ContactData(df_cnt, id_col="part_id", age_col="cnt_age")
pop_data = PopulationData(df_pop, size_col="P", age_col="age")

`SocialMix` can adjust the estimated matrix so that the total number of contacts between two age groups is the same in both directions at the population level (reciprocity, `apply_reciprocity=True`). The adjustment is off by default, since whether it is appropriate depends on the analysis. We leave it off here, which also makes the effect of weighting easier to read. Weighting works in the same way with reciprocity switched on, since the adjustment is applied to the weighted estimate.

In [ ]:
# Since we fit `SocialMix` several times, we wrap the call in a small helper.
# It hides routine data notes (`UserWarning`).
def fit(weight_specs=None, part=None, cnt=None):
    # Fit SocialMix without the reciprocity adjustment, on the tutorial data
    # unless other participant and contact data are given.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        warnings.simplefilter("default", WeightWarning)
        return SocialMix(
            part_data if part is None else part,
            cnt_data if cnt is None else cnt,
            age_group_specs,
            pop_data,
            apply_reciprocity=False,
            weight_specs=weight_specs,
        )


sm = fit()  # the unweighted baseline
plot_mosaic_pixilated(
    sm.cint()["All->All"],
    title="Unweighted Contact Matrix",
    zlabel="Intensity",
    xlabel="Age of respondent",
    ylabel="Age of contact",
)

## 3. Day-of-week weights

To understand how weights are defined and constructed, we begin with a simple example. Suppose we wish to estimate the average contact intensity over a typical week. A week has five weekdays and two weekend days, so within each age group, reports from weekdays and weekends should contribute in the ratio 5:2.

Let $\mathcal{N}_c$ be the set of participants in age group $c$, and let $\mathcal{W}$, $\mathcal{E}$ and $\mathcal{M}$ be the sets of participants who reported on a weekday, on a weekend day, and with a missing day, respectively. In age group $c$, let $n_{c,\text{wd}} = \lvert \mathcal{N}_c \cap \mathcal{W} \rvert$ and $n_{c,\text{we}} = \lvert \mathcal{N}_c \cap \mathcal{E} \rvert$ be the numbers of participants who reported on a weekday and on a weekend day. Each participant $i$ receives the target share of their day type divided by its share among the participants of their age group with a known day:
$$
f^{\text{dow}}_i =
\begin{cases}
\dfrac{5/7}{n_{c,\text{wd}} / (n_{c,\text{wd}} + n_{c,\text{we}})}, & i \in \mathcal{N}_c \cap \mathcal{W}, \\[12pt]
\dfrac{2/7}{n_{c,\text{we}} / (n_{c,\text{wd}} + n_{c,\text{we}})}, & i \in \mathcal{N}_c \cap \mathcal{E}, \\[12pt]
1, & i \in \mathcal{N}_c \cap \mathcal{M}.
\end{cases}
$$
A day type that is under-represented in an age group receives a factor above 1, and an over-represented one a factor below 1. Among the participants with a known day, the factors average 1, so participants with a missing day receive the average factor.

In `cntmosaic`, we specify weights through the `WeightSpecs` class and pass it to `SocialMix`. `WeightSpecs` takes various factor classes as inputs. We will describe factor classes in more detail in the following sections. Suffice it to say that they tell the model which variables we want to weight the matrix by and how. In this case, because we want to weight by day-of-week, we use the predefined `DayOfWeekWeights` class.

Additionally, before fitting the model, `WeightSpecs.preview()` shows the weights that the fit would use.

In [ ]:
dow_specs = WeightSpecs(DayOfWeekWeights())
dow_weights = dow_specs.preview(part_data, age_group_specs)
dow_weights.participants.head()

The `participants` table has one row per participant, with the following columns:

- `cell`: the participant's cell, i.e. their age group combined with any participant strata used in the fit (e.g. sex). This fit has no participant strata, so every cell is labelled `All` followed by the age group. In this tutorial, the cells are therefore the age groups;
- `part_dayofweek`: the participant column the factor is computed from (each factor's source column is shown once, under its `part_` name);
- `dayofweek`: the factor $f^{\text{dow}}_i$ (each column is labelled by the factor's `name`: `dayofweek` and `age` for the two ready-made factors, `poststrat:<col>`, `design:<col>` or `category:<col>` for the others; another label can be set with `name=`);
- `composite`: the product of all factors, $\tilde w_i$, which here equals the single factor;
- `weight`: the final weight $w_i$, i.e. the composite weight rescaled so that the weights within each age group sum to the number of participants in it (Section 5);
- `capped`: whether the weight was truncated (Section 8).

The factor takes one value per age group and day type:

In [ ]:
# Display only: participants and factors by age group and day type
participants = dow_weights.participants.assign(
    day_type=day_type.loc[dow_weights.participants["id"]].to_numpy()
)
age_groups = dow_weights.cells["cell"]  # age groups, in order
factor_table = pd.concat(
    {
        "participants": participants.pivot_table(
            index="cell", columns="day_type", values="id", aggfunc="size"
        ),
        "factor": participants.pivot_table(
            index="cell", columns="day_type", values="dayofweek", aggfunc="first"
        ),
    },
    axis=1,
).reindex(age_groups)
factor_table.index = factor_table.index.str.removeprefix("All, ")
factor_table.round(3).head(4)

For example, the under-5s have 59 weekday and 28 weekend reports, i.e. a weekday share of 0.678 among those with a known day, below 5/7. Their weekday reports are therefore upweighted (factor 1.053) and their weekend reports downweighted (0.888). The 5–9-year-olds have the opposite imbalance, with 75 weekday and 22 weekend reports, so their weekend reports are upweighted (1.260).

How do the weights enter the contact matrix? Let $y_{id}$ be the number of contacts in age group $d$ reported by participant $i$. `SocialMix` estimates $m_{cd}$, the mean number of contacts that a participant in age group $c$ has with people in age group $d$, as a weighted mean over the participants in age group $c$:

<a id="weighted-mean"></a>
$$
m_{cd} \;=\; \frac{\sum_{i \in \mathcal{N}_c} w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} w_i}. \tag{1}
$$

With the factors above, the participants of age group $c$ who reported on a weekday and those who reported on a weekend day contribute to $m_{cd}$ in the ratio 5:2.

## 4. Matching 5:2 within age groups or across the sample?

**In short:** use the default `within="cell"`, which meets the 5:2 ratio in every age group. Consider `within="sample"` only when some age groups have very few weekday or weekend reports. With participant strata, e.g. sex, the cells are age group × sex: `within="cell"` then matches the 5:2 ratio in every age group of each sex, and the counts per cell are smaller.

`DayOfWeekWeights` matches the 5:2 ratio within each age group, which is the default, `within="cell"`. The R package socialmixr instead computes the factors from the counts over the whole sample. In `cntmosaic`, this corresponds to `within="sample"`:
$$
f^{\text{dow}}_i = \frac{5/7}{n_{\text{wd}} / (n_{\text{wd}} + n_{\text{we}})} \;\; (i \in \mathcal{W}),
\qquad
f^{\text{dow}}_i = \frac{2/7}{n_{\text{we}} / (n_{\text{wd}} + n_{\text{we}})} \;\; (i \in \mathcal{E}),
$$
with a factor of 1 for a missing day, where $n_{\text{wd}} = \lvert \mathcal{W} \rvert$ and $n_{\text{we}} = \lvert \mathcal{E} \rvert$ are counted over the whole sample.

To see the consequence, we split $\mathcal{N}_c$ by day type, leaving out the participants with a missing day for clarity. Let $\bar y^{\text{wd}}_{cd}$ and $\bar y^{\text{we}}_{cd}$ be the unweighted mean contacts with age group $d$ of the participants in age group $c$ who reported on a weekday and on a weekend day:

$$
\bar y^{\text{wd}}_{cd} = \frac{1}{n_{c,\text{wd}}} \sum_{i \in \mathcal{N}_c \cap \mathcal{W}} y_{id},
\qquad
\bar y^{\text{we}}_{cd} = \frac{1}{n_{c,\text{we}}} \sum_{i \in \mathcal{N}_c \cap \mathcal{E}} y_{id}.
$$

Since all weekday participants of age group $c$ share one factor, $f^{\text{wd}}$, and all weekend participants share another, $f^{\text{we}}$, substituting into [(1)](#weighted-mean) gives a mixture of the two day-type means:

$$
m_{cd} \;=\; \lambda^{\text{wd}}_c \, \bar y^{\text{wd}}_{cd} \;+\; \lambda^{\text{we}}_c \, \bar y^{\text{we}}_{cd},
\qquad
\lambda^{\text{wd}}_c = \frac{n_{c,\text{wd}} \, f^{\text{wd}}}{n_{c,\text{wd}} \, f^{\text{wd}} + n_{c,\text{we}} \, f^{\text{we}}},
\qquad
\lambda^{\text{we}}_c = 1 - \lambda^{\text{wd}}_c .
$$

We refer to $\lambda^{\text{wd}}_c$ as the *weighted weekday share* of age group $c$. With the factors computed within each age group, $\lambda^{\text{wd}}_c = 5/7$ in every age group. With the factors computed over the whole sample,
$$
\lambda^{\text{wd}}_c
\;=\; \frac{n_{c,\text{wd}} \cdot 5 / n_{\text{wd}}}{n_{c,\text{wd}} \cdot 5 / n_{\text{wd}} \;+\; n_{c,\text{we}} \cdot 2 / n_{\text{we}}},
$$
which equals $5/7$ only when $n_{c,\text{wd}} / n_{c,\text{we}} = n_{\text{wd}} / n_{\text{we}}$, i.e. when age group $c$ has the same weekday-to-weekend ratio as the whole sample. The sample-wide factors therefore achieve 5:2 across the whole sample, but not within each age group. The weekday share of every age group is shifted by the same odds ratio, so an age group whose day mix already deviates from 5:2 in the same direction as the shift is pushed further away from it.

The table below compares, for each age group, the weighted weekday share and the total contact intensity without weights and with the two options.

In [ ]:
sm_dow = fit(dow_specs)
M_dow = sm_dow.cint()["All->All"]  # weighted weekly average

sample_specs = WeightSpecs(DayOfWeekWeights(within="sample"))
sample_weights = sample_specs.preview(part_data, age_group_specs)
M_sample = fit(sample_specs).cint()["All->All"]
M_raw = sm_dow.cint(weighted=False)["All->All"]  # unweighted, from the same fit

# Display only: weekday share and total intensity by age group, for each variant
ids = dow_weights.participants["id"]
cells = dow_weights.participants["cell"]  # age group of each participant
days = day_type.loc[ids].to_numpy()


def weekday_share(w):
    # Weighted weekday share among participants with a known day, per age group
    w = pd.Series(np.where(days == "missing", 0.0, w), index=cells.index)
    share = (w * (days == "weekday")).groupby(cells).sum() / w.groupby(cells).sum()
    return share.reindex(age_groups).to_numpy()


within_table = pd.DataFrame(
    {
        "weekday share: unweighted": weekday_share(1.0),
        "weekday share: sample": weekday_share(
            sample_weights.participants.set_index("id").loc[ids, "weight"].to_numpy()
        ),
        "weekday share: cell": weekday_share(dow_weights.participants["weight"]),
        "intensity: unweighted": M_raw.central.sum(axis=1),
        "intensity: sample": M_sample.central.sum(axis=1),
        "intensity: cell": M_dow.central.sum(axis=1),
    },
    index=age_groups.str.removeprefix("All, "),
)
within_table.round(3)

With `within="sample"`, an age group that already has too few weekday reports is pushed further from 5:2, and its estimate moves in the wrong direction. The whole sample over-represents weekdays (73.6% against 71.4%), so the sample-wide factors shift the weekday share of every age group down by the same odds ratio. For the under-5s, whose share is already below 5/7, it falls from 0.678 to 0.654, and their contact intensity decreases from 8.94 to 8.84, whereas a weekly average should increase it. With `within="cell"`, the weekday share is exactly 5/7 in every age group, and the contact intensity of the under-5s increases to 9.08.

The sample-wide factors are nevertheless not without merit. They are computed from the counts of the whole sample, which are large, so the factors are stable. The factors computed within each age group depend on $n_{c,\text{wd}}$ and $n_{c,\text{we}}$, which may be small. When one of them is small, the few participants concerned receive large weights, which increases the variance of the estimate (Section 8). When $n_{c,\text{we}} = 0$ (or $n_{c,\text{wd}} = 0$), the factor of the absent day type is undefined, and the 5:2 ratio cannot be achieved in that age group: whatever the weights, the estimate reduces to the mean of the observed day type. `cntmosaic` issues a `WeightWarning` in this case. In POLYMOD, every age group contains both day types, so the warning does not arise here. When age groups are small, it would be prudent to merge them, or to use `within="sample"` and accept that the 5:2 ratio then holds only across the whole sample.

## 5. How does weighting work?

In this section, we describe how weights are constructed in general. The day-of-week example above already contains all the ingredients of the general setup. `SocialMix` computes the weighted estimate in three steps.

*Step 1: combining factors.* A `WeightSpecs` holds one or more weight factors $k = 1, \dots, K$. Each gives participant $i$ a factor $f^{(k)}_i$, and the factors are multiplied to create a composite weight
$$
\tilde w_i = \prod_{k=1}^{K} f^{(k)}_i .
$$

*Step 2: normalisation.* The composite weights are rescaled so that they sum to the number of participants in each age group, $n_c = \lvert \mathcal{N}_c \rvert$:
$$
w_i \;=\; \tilde w_i \cdot \frac{n_c}{\sum_{j \in \mathcal{N}_c} \tilde w_j}, \quad i \in \mathcal{N}_c,
\quad\text{so that} \quad \sum_{i \in \mathcal{N}_c} w_i = n_c .
$$
On this scale, the average weight of every age group $c$ is 1. This is particularly useful if we want to truncate the weights (Section 8) with a common threshold that is applied across age groups.

*Step 3: estimation.* Recall the weighted mean [(1)](#weighted-mean), computed separately for each age group $c$:

$$
m_{cd} \;=\; \frac{\sum_{i \in \mathcal{N}_c} w_i \, y_{id}}{\sum_{i \in \mathcal{N}_c} w_i}.
$$

Rescaling the weights within an age group does not change the estimate. For any constant $\gamma_c > 0$, replacing $w_i$ by $\gamma_c w_i$ for all $i \in \mathcal{N}_c$ leaves $m_{cd}$ unchanged, since $\gamma_c$ cancels between the numerator and the denominator. Normalisation is such a rescaling, so without truncation the composite weights $\tilde w_i$ give the same estimate as the final weights $w_i$.

> **Consequences of normalisation**
>
> 1. Only the ratios between factors within an age group matter. In Section 3, only the ratio of the weekday and weekend factors matters, not their absolute size.
> 2. Weights do not change the relative size of the age groups, since the weights of each age group sum to its own $n_c$.
> 3. The population data given to `SocialMix` does not enter $m_{cd}$. It is used only afterwards, for the reciprocity adjustment or for per-capita contact rates. A population can still enter through a weight factor, e.g. `AgePostStratWeights`, but only through the relative weights within each age group (consequences 1 and 2).
> 4. A factor that is constant within every age group has no effect, e.g. a weight defined on the age group itself.
> 5. The order of the factors does not matter, since multiplication commutes.
>
> Truncation is the only step in which normalisation changes the estimate (Section 8).

**What the weights estimate.** With weights, $m_{cd}$ estimates the mean number of contacts with age group $d$ of a participant in age group $c$, averaged over the composition given by the targets (e.g. over a typical week) rather than over the composition of the sample. In survey statistics, (1) is a ratio (Hájek) estimator, and with factors computed from the sample counts it is a post-stratified estimator, i.e. one whose weights match the sample composition to known population shares. It corrects the composition of the sample, but not who responded within a category: the weekend reporters of age group $c$, for instance, are assumed to have representative weekend contact behaviour in that age group. Participants with a missing value receive the average factor, so their own composition is not corrected.

## 6. Which weight factor should I use?

`cntmosaic` provides five weight factors. The appropriate choice depends on what is known about the sample:

| What is known                                                                                                                                                | Factor                           | Total factor of a category                                     |
|--------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------|----------------------------------------------------------------|
| a weight for each participant, supplied with the survey (inverse inclusion probabilities, non-response adjustments)                                          | `DesignWeights(col)`             | the sum of the supplied weights                                |
| a fixed weight for each category, set by the design rather than by the number of participants sampled in it (e.g. the inverse sampling fraction of a region) | `CategoryWeights(col, weights)`  | factor × number sampled, i.e. it *grows* with the sample count |
| the share that each category should have (a population distribution, the 5:2 split of weekdays and weekends, chosen country shares)                          | `PostStratWeights(col, targets)` | the target share, *whatever* the sample count                  |

`DayOfWeekWeights` and `AgePostStratWeights` are ready-made post-stratifications for the two most common cases.

A simple test helps to decide between `CategoryWeights` and `PostStratWeights`. Suppose twice as many participants had been sampled in category X. If the total influence of X should then double, use `CategoryWeights`. If it should remain at its target share, use `PostStratWeights`.

Both classes accept the table either as a dict, e.g. `{"urban": 1.0, "rural": 2.5}`, or as a two-column DataFrame (`key_col=`, `value_col=`), which is convenient when the table is read from a file. Every value observed in the data must appear in the table: an unknown value raises an error rather than producing a missing weight.

The post-stratification factors depend on how many participants fall in each category. The option `within=` determines where these counts are taken: over the whole sample (`"sample"`) or within each age group (`"cell"`). With `"cell"`, the same targets are applied in every age group, which is appropriate only when the targets hold within each age group, e.g. the 5:2 ratio of weekdays to weekend days. It is not appropriate for a composition that varies with age, e.g. the sex ratio of the national population. Hence the defaults differ between factors:

| Factor                             | Uses the sample counts | Default `within`                                                     |
|------------------------------------|------------------------|----------------------------------------------------------------------|
| `DesignWeights`, `CategoryWeights` | no                     | not applicable                                                       |
| `PostStratWeights`                 | yes                    | `"sample"`                                                           |
| `DayOfWeekWeights`                 | yes                    | `"cell"`                                                             |
| `AgePostStratWeights`              | yes                    | `"sample"` (the same weights as `"cell"` without participant strata) |

## 7. The weight factors

### `DayOfWeekWeights`

See Sections 3 and 4. Its arguments, the data it expects (column names, day codes, missing values) and the errors it raises for unexpected codes are described in the [Appendix](#Appendix:-factor-arguments-and-data-assumptions).

### `AgePostStratWeights`

For each single-year age $a$ in the sample, let $n_a$ be the number of participants of age $a$ and $P_a$ the population of that age, and let $P$ be the population summed over the ages present in the sample and $N$ the number of participants. The factor of participant $i$ is the population share of their age divided by its sample share:

$$
f^{\text{age}}_i \;=\; \frac{P_{a} / P}{n_{a} / N}, \qquad a = \text{the single-year age of participant } i .
$$

Only the variation of this factor within each age group affects the estimate (Section 5). Within age group $c$, the weight is proportional to $P_a / n_a$, so age post-stratification rebalances the single years of age within each age group: over-sampled ages are downweighted relative to under-sampled ones. With one age group per year of age it has no effect, and `cntmosaic` issues a warning.

As an example, consider the weights in the age group $[65, 70)$:

In [ ]:
age_specs = WeightSpecs(AgePostStratWeights(pop_data))
age_weights = age_specs.preview(part_data, age_group_specs)

(
    age_weights.participants.query("cell == 'All, [65, 70)'")
    .assign(year=lambda t: t["part_age"].astype(int))
    .groupby("year")
    .agg(n_a=("id", "size"), factor=("age", "first"), weight=("weight", "first"))
    .assign(P_a=lambda t: df_pop.set_index("age").loc[t.index, "P"].to_numpy())
    .round(3)
)

The population is spread fairly evenly over ages 65–69, but the sample is not: 13 participants are aged 65 and only one is aged 66. The single 66-year-old therefore receives a final weight of 5.2, i.e. the weight of about five average participants in the age group. Such large weights make the estimate depend heavily on a few individuals. Section 8 shows how to detect and limit them.

The handling of banded and open-ended population data is described in the [Appendix](#Appendix:-factor-arguments-and-data-assumptions).

We fit the age post-stratified model and compare it with the unweighted estimate:

In [ ]:
M_age = fit(age_specs).cint()["All->All"]

alt.hconcat(
    plot_mosaic_pixilated(
        M_age,
        title="Age post-stratified",
        zlabel="Intensity",
        xlabel="Age of respondent",
        ylabel="Age of contact",
    ),
    plot_mosaic_pixilated(
        M_age.central - M_raw.central,
        title="Age post-stratified minus unweighted",
        xlabel="Age of respondent",
        age_group_specs=age_group_specs,
        zlabel="difference",
        show_y_label=False,
        show_y_ticks=False,
        color_scheme="redblue",
        color_mid=0,
    ),
).resolve_scale(color="independent")

### `PostStratWeights`

`PostStratWeights(col, targets)` matches the weighted composition of any participant variable to a set of target shares. The targets may be given as totals, shares or population counts, since only their ratios matter. For example, we may match the sex ratio of the UK population. The targets should describe the population that the sample represents, so we restrict the census to ages 0–79, the range covered by `age_group_specs`. Sex varies within every age group, so these weights affect the estimate.

In [ ]:
# Restrict the census to the ages the sample covers (0-79)
df_uk = pd.read_csv("data/UK_population_2011.csv")
df_uk = df_uk[~df_uk["age"].str.endswith("+")]
df_uk = df_uk[df_uk["age"].astype(int) <= 79]
sex_targets = {"M": df_uk["Male"].sum(), "F": df_uk["Female"].sum()}

sex_weights = WeightSpecs(PostStratWeights("gender", sex_targets)).preview(
    part_data, age_group_specs
)
sex_weights.participants.groupby("part_gender", observed=True)[
    "poststrat:gender"
].first()

Women make up 51.8% of the sample but 50.3% of the population aged 0–79, and are therefore slightly downweighted. Had we used the whole census, the target would have been 50.9%, since older age groups contain more women than men.

Its further arguments (`groups=`, `on_missing=`, `within=`, and targets given as a DataFrame) are described in the [Appendix](#Appendix:-factor-arguments-and-data-assumptions).

### `DesignWeights` and `CategoryWeights`

Some surveys provide design weights, which correct for unequal probabilities of selection. POLYMOD does not, so we construct a synthetic example. We randomly assign each participant to an `"urban"` or a `"rural"` region, and suppose that rural households were sampled at half the rate of urban ones. Each rural participant then represents twice as many people, so their weight is 2.

There are two ways of applying such weights:

1. if the survey provides a weight for each participant, stored as a column (here `design_w`), we use `DesignWeights`;
2. if the weight of each category is known, we use `CategoryWeights` to look it up.

Note that the weight variable must vary within age groups to affect the estimate (Section 5). A randomly assigned region does.

In [ ]:
rng = np.random.default_rng(2008)
df_synth = df_part.copy()
df_synth["region"] = rng.choice(["urban", "rural"], size=len(df_synth), p=[0.8, 0.2])
df_synth["design_w"] = df_synth["region"].map({"urban": 1.0, "rural": 2.0})

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    part_synth = ParticipantData(
        df_synth, id_col="part_id", age_col="part_age", aux_cols=["region", "design_w"]
    )

by_participant = WeightSpecs(DesignWeights("design_w")).preview(
    part_synth, age_group_specs
)
by_category = WeightSpecs(
    CategoryWeights("region", {"urban": 1.0, "rural": 2.0})
).preview(part_synth, age_group_specs)

pd.DataFrame(
    {
        "region": by_category.participants["part_region"],
        "DesignWeights": by_participant.participants["weight"],
        "CategoryWeights": by_category.participants["weight"],
    }
).head()

Both give the same weights here, since the design weight depends on region alone. In practice, design weights usually combine several sampling stages and non-response adjustments, and vary from participant to participant. `DesignWeights` is then the only option.

### Combining factors

Several factors can be combined in one `WeightSpecs`, e.g. `WeightSpecs(DayOfWeekWeights(), AgePostStratWeights(pop_data))`, in which case they are multiplied (Section 5, step 1).

It is important to note that post-stratification factors are not calibrated jointly. Each factor matches its own target on its own. When the factors are correlated, e.g. when the day mix differs between single years of age, multiplying them may move each composition away from its target. The day-of-week factor alone puts the weekday share of the whole sample at exactly 5/7, but this share drifts once the age factor is included:

In [ ]:
both_specs = WeightSpecs(DayOfWeekWeights(), AgePostStratWeights(pop_data))
both = both_specs.preview(part_data, age_group_specs).participants
both_days = day_type.loc[both["id"]].to_numpy()
known = both_days != "missing"

for column, label in [
    ("dayofweek", "day-of-week factor alone"),
    ("composite", "day-of-week x age"),
]:
    w = both[column].to_numpy()[known]
    print(
        f"Weekday share, {label}: {np.sum(w * (both_days[known] == 'weekday')) / np.sum(w):.4f}"
    )
print(f"Target 5/7: {5 / 7:.4f}")

The drift is small here (from 0.714 to 0.708), but it may grow with more factors and stronger weights. Matching several compositions simultaneously is known as raking, which is not available in `cntmosaic`. Raked weights can be computed with survey software, e.g. `rake()` in the R package survey, and passed to `SocialMix` with `DesignWeights`. They should then be the only factor, since multiplying them by further factors would undo the calibration.

## 8. Inspecting and stabilising weights

Weights can be inspected both before and after fitting. Before fitting, `WeightSpecs.preview(part_data, age_group_specs)` computes the weights without contact data. It uses the same age groups as the fit, including any merging of empty age groups with `adaptive_merge=True`, which makes it useful for checking the factors and spotting extreme weights. After fitting, a weighted `SocialMix` stores the weights it used in `sm.weights`, a `WeightResult` with two tables:

1. `participants`, with one row per participant, as in Section 3;
2. `cells`, with one row per age group, giving the number of participants `n`, the smallest and largest weight, the number of capped weights, the effective sample size `n_eff` and the design effect `deff`.

Unequal weights make an estimate less precise than an unweighted estimate from the same number of participants. Kish's effective sample size measures this loss:

$$
n_{\text{eff},c} \;=\; \frac{\left(\sum_{i \in \mathcal{N}_c} w_i\right)^2}{\sum_{i \in \mathcal{N}_c} w_i^2},
\qquad
\text{deff}_c = \frac{n_c}{n_{\text{eff},c}} .
$$

With equal weights, $n_{\text{eff},c} = n_c$ and the design effect is 1. A design effect of 2 means that the weighted estimate for that age group is roughly as precise as an unweighted estimate from half as many participants.

In [ ]:
sm_both = fit(both_specs)
print(sm_both.weights)
sm_both.weights.cells.round(2)

The day-of-week factor costs almost no precision: the design effects of the larger age groups remain close to 1.05. The age factor matters more in age groups with uneven single-year counts. In $[65, 70)$ the design effect is about 2, so its 25 participants are worth about 12.

Extreme weights allow a few participants to dominate an age group. Setting `WeightSpecs(..., threshold=τ)` limits them in two steps, applied after normalisation (Section 5, step 2):

1. cap the weights: $w_i \leftarrow \min(w_i, \tau)$;
2. re-normalise once: $w_i \leftarrow w_i \cdot n_c \big/ \sum_{j \in \mathcal{N}_c} w_j$ for $i \in \mathcal{N}_c$.

The threshold is on the normalised scale, on which the mean weight of each age group is 1, so `threshold=3` caps every weight at three times the average. Since re-normalisation is performed only once, weights close to $\tau$ may end slightly above it. Truncation is the only step in which normalisation changes the estimate: it gives up part of the correction (bias) in exchange for stability (variance).

In [ ]:
capped_specs = WeightSpecs(
    DayOfWeekWeights(), AgePostStratWeights(pop_data), threshold=3
)
capped = capped_specs.preview(part_data, age_group_specs)
print(capped)

uncapped_cells = sm_both.weights.cells.set_index("cell")
capped_cells = capped.cells.set_index("cell")
pd.DataFrame(
    {
        "max weight": uncapped_cells["max_w"],
        "max weight, capped": capped_cells["max_w"],
        "n capped": capped_cells["n_capped"],
        "deff": uncapped_cells["deff"],
        "deff, capped": capped_cells["deff"],
    }
).query("`n capped` > 0").round(2)

Four participants were capped, in $[55, 60)$ and $[65, 70)$. The largest weight falls from 5.22 to 3.29, slightly above the threshold for the reason given above, and the design effect of $[65, 70)$ falls from 2.06 to 1.61.

Weighting messages use their own warning category, `WeightWarning` (a subclass of `UserWarning`), so that they can be filtered separately from routine data notes. For example, `cntmosaic` warns when a factor is constant within every age group and therefore cannot change the contact matrix. This is the case for age post-stratification with one age group per year of age:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", WeightWarning)
    single_year = AgeGroupSpecs(0, 80, 1)  # one age group per year of age (0-79)
    WeightSpecs(AgePostStratWeights(pop_data)).preview(part_data, single_year)

[str(w.message) for w in caught if issubclass(w.category, WeightWarning)]

The first message is unrelated to weighting: no participant is aged 76, so a fit with these age groups would require `adaptive_merge=True`, and `preview()` reports the empty age group instead of raising an error. The second message is the one we expected.

## 9. Uncertainty with the weighted bootstrap

`run_inference_bootstrap()` quantifies uncertainty by resampling participants with replacement and re-estimating the contact matrix on each resample. Participants are resampled from the whole sample, not within each age group, so the age-group sizes $n_c$ also vary between resamples. The 95% intervals are percentile intervals, i.e. the 2.5% and 97.5% quantiles of the resampled estimates. For a weighted fit, the weights are recomputed on every resample. The factors depend on sample counts such as $n_{c,\text{wd}}$, $n_{c,\text{we}}$ and $n_a$, which vary between resamples, and this variability should be reflected in the intervals. The unweighted bootstrap of the same fit is obtained with `run_inference_bootstrap(weighted=False)`, which ignores the weights entirely, as if the model had been fitted without `weight_specs`.

If an age group is too small for stable resampling, the bootstrap merges it with a neighbouring age group. The weights are then recomputed on the merged age groups, and a `WeightWarning` notes that `sm.weights` and the weighted point estimate have changed. This happens here: the 21 participants aged 70–74 and the 8 aged 75–79 are merged into $[70, 80)$. An age group is merged when it is so small that a resample could contain none of its participants (the exact rule is given in the docstring of `run_inference_bootstrap`). The merge persists after the bootstrap, so the point estimate should be taken from the model *after* bootstrapping, and the merged age groups should be reported. To control the grouping yourself, define the age groups with the merge already made before fitting, e.g. `AgeGroupSpecs(0, 79, cuts=list(range(5, 75, 5)))` for five-year age groups up to 70 and a single age group 70–79.

In [ ]:
sm_boot = fit(dow_specs)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    warnings.simplefilter("always", WeightWarning)
    boot = sm_boot.run_inference_bootstrap(n_boot=500, random_state=1, progress=False)

print(boot)
print(sm_boot.weights)

Weights usually make an estimate less precise. To illustrate this, we bootstrap four specifications (unweighted, day-of-week, day-of-week × age, and day-of-week × age with the weights capped at 3) and compare the mean width of the 95% intervals in each age group.

In [ ]:
specifications = {
    "unweighted": None,
    "day": dow_specs,
    "day x age": both_specs,
    "day x age, capped": capped_specs,
}

summaries, widths = {}, {}
for label, specs in specifications.items():
    model = fit(specs)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # the merge is reported above
        model.run_inference_bootstrap(n_boot=500, random_state=1, progress=False)
    summaries[label] = ModelSummariserSocialMix(model).summarise_cint()["All->All"]
    widths[label] = np.mean(summaries[label].upper - summaries[label].lower, axis=1)

merged_age_groups = sm_boot.weights.cells["cell"].str.removeprefix("All, ")
width_table = pd.DataFrame(widths, index=merged_age_groups)
width_table.loc["mean"] = width_table.mean()
width_table.round(3)

The day-of-week weights leave the average width unchanged (0.556 against 0.557), as they are close to 1. They narrow some intervals and widen others, most of all $[70, 80)$. Including the age factor widens the intervals by about 9% on average, and most in $[55, 60)$ and $[65, 70)$, the age groups with the largest design effects in Section 8. The design effect is a guide rather than a prediction: $[30, 35)$ has a design effect of 1.34 and its interval does not widen, whereas $[45, 50)$ widens by 12% with a design effect of 1.19. The design effect measures only how unequal the weights are, not how the weights relate to the number of contacts. Capping the weights at 3 recovers part of the lost precision. With 500 bootstrap replicates, differences of a few percent between individual age groups may be Monte Carlo error.

## 10. Should we weight or stratify?

The matrix weighted by day of week (Section 3) estimates the contact intensity on an average day of the week. This is the appropriate target when the quantity of interest is itself an average over the whole week.

However, if we are interested in the contacts that occur at school and at the workplace, and in the role they play in transmission, the weighted weekly average is not the appropriate target. These contacts occur mainly on weekdays. Weighting does not isolate them: it mixes weekday and weekend contact behaviour in the ratio 5:2, and therefore dilutes weekday contacts with weekend contacts. In this case, it would be prudent to stratify the data by day type instead, and to estimate the matrix from the participants who reported on a weekday only.

To illustrate the difference, we subset the data to the participants who reported on a weekday and compare the resulting matrix with the weighted weekly average.

In [ ]:
def subset(ids):
    # Participant and contact data restricted to the participants in `ids`.
    # The routine data notes (`UserWarning`) were shown in Section 2.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        part = ParticipantData(
            df_part[df_part["part_id"].isin(ids)],
            id_col="part_id",
            age_col="part_age",
            aux_cols=["dayofweek", "part_gender"],
        )
        cnt = ContactData(
            df_cnt[df_cnt["part_id"].isin(ids)], id_col="part_id", age_col="cnt_age"
        )
    return part, cnt


part_wd, cnt_wd = subset(day_type.index[day_type == "weekday"])
M_wd = fit(part=part_wd, cnt=cnt_wd).cint()["All->All"]  # weekday reports only

alt.hconcat(
    plot_mosaic_pixilated(
        M_wd,
        title="Weekday reports only",
        xlabel="Age of respondent",
        ylabel="Age of contact",
    ),
    plot_mosaic_pixilated(
        M_dow,
        title="Weighted weekly average",
        xlabel="Age of respondent",
        zlabel="Intensity",
        show_y_label=False,
        show_y_ticks=False,
    ),
    plot_mosaic_pixilated(
        M_wd.central - M_dow.central,
        age_group_specs=age_group_specs,
        title="Weekday minus weekly average",
        xlabel="Age of respondent",
        zlabel="difference",
        show_y_label=False,
        show_y_ticks=False,
        color_scheme="redblue",
        color_mid=0,
    ),
).resolve_scale(color="independent")

To compare the two estimates by age group, we look at the total contact intensity of each age group, i.e. the row sums $\sum_d m_{cd}$. For reference, the table also gives the estimate from the participants who reported on a weekend day.

In [ ]:
part_we, cnt_we = subset(day_type.index[day_type == "weekend"])
M_we = fit(part=part_we, cnt=cnt_we).cint()["All->All"]  # weekend reports only

intensity_table = pd.DataFrame(
    {
        "weekday only": M_wd.central.sum(axis=1),
        "weekend only": M_we.central.sum(axis=1),
        "weighted weekly average": M_dow.central.sum(axis=1),
    },
    index=age_groups.str.removeprefix("All, "),
)
intensity_table.round(2).head(4)

The weighted weekly average combines the weekday and weekend reports in the ratio 5:2, together with the reports with a missing day. It therefore usually lies between the weekday and the weekend estimates, as in the age groups shown. For school-age children, the weekday estimate is considerably higher than the weekly average: 15.28 against 14.25 for the 5–9-year-olds, and 17.44 against 14.93 for the 10–14-year-olds. If the analysis concerns transmission during the school term, e.g. to assess the effect of school closures, the weighted weekly average would therefore understate the contact intensity of children. This may in turn underestimate key epidemiological quantities such as $R_0$.

Note that POLYMOD does not record whether the survey day fell in a school holiday, so some weekday reports may come from holidays. Where such information is available, it may be used to restrict the data further.

## 11. Limitations and checklist

The following are not supported by `cntmosaic`:

1. contact-level weights: all weights are participant-level, and each contact inherits the weight of the participant who reported it;
2. raking, i.e. calibration on several compositions at once, e.g. age × sex × region; raked weights computed elsewhere can be passed with `DesignWeights` (Section 7);
3. design-based variance estimation for complex samples with strata or clusters: the bootstrap resamples participants independently, and replicate weights supplied with a survey (e.g. jackknife or bootstrap replicate weights) cannot be used;
4. weights in the Bayesian models: `weight_specs` is available for `SocialMix` only, not for `Prem2` or the GenMix models.

Before reporting a weighted contact matrix, we suggest checking the following:

1. Is a weighted average the appropriate target, or should the data be stratified instead, e.g. by day type (Section 10)?
2. Do both conditions of Section 1 hold? That is, do we wish to average over a factor that influences contact behaviour, and does the sample composition differ from the population composition for that factor?
3. Does the weighting variable vary within age groups? Otherwise it has no effect (Section 5).
4. Are the targets correct, and are the codes mapped correctly, e.g. the day coding (Appendix)?
5. Are the counts taken where the targets hold (`within`, Sections 4 and 6)? Did any age group lack a category, e.g. a day type, so that its target could not be met?
6. How are missing values treated (`on_missing`, Appendix)?
7. How large are the design effects, and is truncation needed (Section 8)?
8. Were the intervals computed with the weighted bootstrap (Section 9)?

A methods section might report the weighting as follows, adapted to the choices made:

> *Participants were weighted so that weekday and weekend reports contribute in the ratio 5:2 within each five-year age group, and post-stratified on single-year age to the 2011 UK census population. Participants with an unrecorded survey day received the average weight. Weights were normalised within age groups and truncated at three times the age-group mean. 95% intervals were obtained as percentiles of 500 bootstrap resamples of participants, with the weights recomputed in each resample; for the bootstrap, the age groups 70–74 and 75–79 were merged.*

## 12. Summary and further reading

For a weekly-average matrix from a diary survey, a typical starting point is `DayOfWeekWeights()`, adding `AgePostStratWeights` if the single years of age within age groups are unevenly sampled, and checking the design effects (Section 8) before deciding on truncation.

The following code summarises the weighting interface:

```python
from cntmosaic.weights import (
    WeightSpecs, DayOfWeekWeights, AgePostStratWeights,
    PostStratWeights, CategoryWeights, DesignWeights, WeightWarning,
)

# Keep the weighting variables on the participant data
part_data = ParticipantData(df, id_col="pid", age_col="age",
                            aux_cols=["dayofweek", "gender", "region", "design_w"])

specs = WeightSpecs(
    DayOfWeekWeights(),                                   # weekdays : weekend = 5:2 within every age group
    # DayOfWeekWeights(within="sample"),                  # ... across the whole sample only (socialmixr)
    # DayOfWeekWeights(weekday="weekday", weekend="weekend"),      # a labelled day column
    # DayOfWeekWeights(weekday=(1, 2, 3, 4, 5), weekend=(6, 7)),   # 1 = Monday ... 7 = Sunday
    AgePostStratWeights(pop_data),                        # single-year age within age groups
    PostStratWeights("gender", {"M": 0.49, "F": 0.51}),   # any target shares
    CategoryWeights("region", {"urban": 1.0, "rural": 2.0}),   # fixed weight per category
    DesignWeights("design_w"),                            # one weight per participant
    threshold=3,                                          # optional truncation
)

specs.preview(part_data, age_group_specs)                 # inspect before fitting

sm = SocialMix(part_data, cnt_data, age_group_specs, pop_data, weight_specs=specs)
sm.cint()                    # weighted estimate
sm.cint(weighted=False)      # unweighted estimate, same fit
sm.weights.cells             # n_eff and design effects per age group
sm.run_inference_bootstrap() # weights recomputed on every resample
```

**Coming from socialmixr**

The weighting options of the R package socialmixr (version 0.7) correspond to the following factors:

| socialmixr                                                                                     | `cntmosaic`                                         |
|------------------------------------------------------------------------------------------------|-----------------------------------------------------|
| `weigh_by_dayofweek(survey)`, `contact_matrix(weigh_dayofweek = TRUE)`                         | `DayOfWeekWeights(within="sample")`                 |
| `contact_matrix(weigh_age = TRUE)`, `weigh_by_age(survey, pop)` with a single-year `pop`       | `AgePostStratWeights(pop_data)`                     |
| `weigh(survey, "col")` (multiplies a numeric column into the weight)                           | `DesignWeights("col")`                              |
| `weigh(survey, "col", target = <data frame>)` (a factor per value)                             | `CategoryWeights("col", {...})`                     |
| `weigh(survey, "col", target = <named vector>)`, or with `groups =` (a total weight per group) | `PostStratWeights("col", {...})`, or with `groups=` |
| `contact_matrix(weight_threshold = 3)`                                                         | `WeightSpecs(..., threshold=3)`                     |

`AgePostStratWeights` always weights by single-year age. socialmixr's `weigh_by_age()` weights at the age bands of the population it is given, and has no effect when these are the age groups of the matrix.

**References**

- Mossong J, Hens N, Jit M, et al. (2008). Social contacts and mixing patterns relevant to the spread of infectious diseases. *PLoS Medicine* 5(3): e74.
- Kish L (1965). *Survey Sampling.* Wiley.
- Lohr SL (2021). *Sampling: Design and Analysis*, 3rd edition. CRC Press.
- Funk S, Willem L, Gruson H, et al. *socialmixr: Social Mixing Matrices for Infectious Disease Modelling.* R package. <https://github.com/epiforecasts/socialmixr>

## Appendix: factor arguments and data assumptions

### `DayOfWeekWeights`

The arguments are:

| Argument             | Default                     | Meaning                                                                                  |
|----------------------|-----------------------------|------------------------------------------------------------------------------------------|
| `col`                | `"dayofweek"`               | the day column (with or without the `part_` prefix)                                      |
| `weekday`, `weekend` | `(1, 2, 3, 4, 5)`, `(0, 6)` | the day codes of each group, a single code or several (POLYMOD coding: 0 = Sunday)       |
| `ratio`              | `(5, 2)`                    | the targets of weekdays and weekend days                                                 |
| `on_missing`         | `"average"`                 | a missing day receives the average factor 1 (`"average"`) or raises an error (`"raise"`) |
| `within`             | `"cell"`                    | match the ratio within each age group, or across the whole sample (`"sample"`)           |

`DayOfWeekWeights` makes the following assumptions about the data:

1. The survey day is stored in the participant column `col`, named with or without the `part_` prefix, i.e. `"dayofweek"` and `"part_dayofweek"` are equivalent. The column must be kept on the participant data with `ParticipantData(aux_cols=...)`; otherwise, `preview()` and the fit raise a `KeyError`. A column with another name is given through `col`, e.g. `DayOfWeekWeights(col="survey_day")`.
2. The values are matched exactly against the codes in `weekday` and `weekend`. By default, these are the integer codes of POLYMOD, 0 = Sunday, 1 = Monday, ..., 6 = Saturday. An integer column read as floats, e.g. because it contains missing values, is matched as integers (`1.0` matches `1`). Strings are matched as they are: the string `"1"` does not match the integer `1`, and the matching is case-sensitive, so `"Weekday"` does not match `"weekday"`.
3. If the days are coded differently, the codes must be given explicitly. For days coded 1 = Monday, ..., 7 = Sunday, we use `DayOfWeekWeights(weekday=(1, 2, 3, 4, 5), weekend=(6, 7))`. For a column that already holds the labels `"weekday"` and `"weekend"`, we use `DayOfWeekWeights(weekday="weekday", weekend="weekend")`. Labels with varying case should be normalised first, e.g. with `.str.lower()`.
4. A non-missing value that belongs to neither group raises an error that lists the expected codes. A Sunday coded as 7 is therefore never silently given the average factor.
5. Only `NaN`, `None` and `pd.NA` are treated as missing. A placeholder such as `"missing"`, `""` or `-1` is an ordinary value and raises an error, so it must be replaced with `NaN` first.
6. For a pandas `Categorical` column, the observed values are matched; the declared categories play no role.

As an example, we code the survey days as labels, with the placeholder `"missing"`. With the default codes, the labels do not match and an error is raised:

In [ ]:
df_labelled = df_part.assign(survey_day=day_type.loc[df_part["part_id"]].to_numpy())
print(df_labelled["survey_day"].value_counts())

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    part_labelled = ParticipantData(
        df_labelled, id_col="part_id", age_col="part_age", aux_cols=["survey_day"]
    )

try:
    WeightSpecs(DayOfWeekWeights(col="survey_day")).preview(
        part_labelled, age_group_specs
    )
except ValueError as err:
    print(err)

After replacing the placeholder with `NaN` and giving the labels, we obtain the same weights as from the integer codes:

In [ ]:
df_labelled["survey_day"] = df_labelled["survey_day"].replace("missing", np.nan)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    part_labelled = ParticipantData(
        df_labelled, id_col="part_id", age_col="part_age", aux_cols=["survey_day"]
    )

labelled_weights = WeightSpecs(
    DayOfWeekWeights(col="survey_day", weekday="weekday", weekend="weekend")
).preview(part_labelled, age_group_specs)
np.allclose(labelled_weights.participants["weight"], dow_weights.participants["weight"])

### `AgePostStratWeights`

Population data are handled as follows:

1. single-year populations are used directly;
2. banded populations (`age_min`/`age_max` or interval age groups) are spread uniformly over the single years of each band, without rounding;
3. open-ended bands, such as `90+`, are refused when the factor is constructed;
4. a participant whose age has no population raises an error, so that no participant is dropped silently.

### `PostStratWeights`

`PostStratWeights` has further arguments:

- `groups=` lets several values share one target. For example, `PostStratWeights("dayofweek", {"weekday": 5, "weekend": 2}, groups={"weekday": [1, 2, 3, 4, 5], "weekend": [0, 6]}, on_missing="average")` gives the same weights as `DayOfWeekWeights()`.
- `on_missing=` determines the treatment of missing values. The default, `"raise"`, raises an error; `"average"` gives them the factor 1.
- `within=` works as for `DayOfWeekWeights`.

The targets may also be given as a DataFrame: `PostStratWeights("gender", df_targets, key_col="gender", value_col="P")`.